# Phase 0 - data validation

Definition of done for Phase 0: all available sources loaded, joined where possible, with row counts and a feature-overlap table.

Run from the repository root. `main.py` produces the same content as a markdown artifact in `reports/`.

In [1]:
import os
import sys

# Notebooks start in notebooks/; the package is imported as src.shot_quality.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import polars as pl

from src.shot_quality.overlap import build_overlap_table, shared_feature_set
from src.shot_quality.report import (
    make_rate_by_bucket,
    make_rate_by_defender_and_distance,
    summarize_sources,
)
from src.shot_quality.sources.mirror import load_mirror_shots
from src.shot_quality.sources.nba_stats import reachability_report
from src.shot_quality.sources.shot_logs_2015 import load_shot_logs_2015

## 1. Can we reach nba.com at all?

`stats.nba.com` and `cdn.nba.com` are Akamai-protected and refuse datacenter IPs. From a residential connection both should report `ok`; from CI or a cloud VM they will not, and every `nba_api`-backed loader is unusable there.

In [2]:
reachability = reachability_report()
reachability

{'stats.nba.com': 'TimeoutError', 'cdn.nba.com': 'http 403'}

## 2. Load the sources

- 2014-15 shot logs: the only shot-level source with closest-defender distance.
- hoopR mirror: ESPN-derived shot data, a stand-in for `shotchartdetail` when nba.com is unreachable.

If `reachability` above is clean, add `load_shot_chart("2025-26", team_id=...)` here as a third source to confirm the mirror and the official endpoint agree.

In [3]:
sources = {
    "shot_logs_2014_15": load_shot_logs_2015(),
    "hoopr_mirror_2025": load_mirror_shots(2025),
    "hoopr_mirror_2026": load_mirror_shots(2026),
}

summarize_sources(sources)

source,rows,seasons,games,make_rate
str,i64,str,i64,f64
"""shot_logs_2014_15""",128069,"""2015""",904,0.4521
"""hoopr_mirror_2025""",235319,"""2025""",1324,0.4664
"""hoopr_mirror_2026""",236114,"""2026""",1325,0.4674


## 3. Feature overlap

Coverage is the fraction of non-null rows per canonical column. A column that exists but is entirely null counts as unavailable, which is the distinction that decides what Phase 1 may train on.

In [4]:
with pl.Config(tbl_rows=30):
    display(build_overlap_table(sources))

shared_feature_set(sources)

column,shot_logs_2014_15,hoopr_mirror_2025,hoopr_mirror_2026,shared_by_all
str,f64,f64,f64,bool
"""source""",1.0,1.0,1.0,true
"""season""",1.0,1.0,1.0,true
"""game_id""",1.0,1.0,1.0,true
"""period""",1.0,1.0,1.0,true
"""seconds_remaining_in_period""",1.0,1.0,1.0,true
"""shot_distance_ft""",1.0,1.0,1.0,true
"""shot_angle_deg""",0.0,1.0,1.0,false
"""is_three""",1.0,1.0,1.0,true
"""shot_type""",0.0,1.0,1.0,false


['period', 'seconds_remaining_in_period', 'shot_distance_ft', 'is_three']

## 4. Join where possible

The sources share no game or player identifier system (ESPN ids vs. NBA ids, and no overlapping season), so a row-level join is not possible. The meaningful join is on the *shared feature space*: stack the sources on the shared columns and confirm the distributions are comparable enough to train on one and score on the other.

In [5]:
shared_columns = ["source", "season", *shared_feature_set(sources), "made"]
stacked = pl.concat([frame.select(shared_columns) for frame in sources.values()])

stacked.group_by("source").agg(
    [
        pl.len().alias("rows"),
        pl.col("shot_distance_ft").mean().round(2).alias("mean_distance_ft"),
        pl.col("is_three").mean().round(4).alias("three_point_rate"),
        pl.col("made").mean().round(4).alias("make_rate"),
    ]
).sort("source")

source,rows,mean_distance_ft,three_point_rate,make_rate
str,u32,f64,f64,f64
"""hoopr_mirror""",471433,14.62,0.4181,0.4669
"""shot_logs_2014_15""",128069,13.57,0.2647,0.4521


## 5. Sanity checks

Make rate must fall with distance in every source; if it does not, the coordinate transform is wrong.

In [6]:
for name, frame in sources.items():
    print(name)
    display(make_rate_by_bucket(frame, "shot_distance_ft", [0.0, 4.0, 10.0, 16.0, 22.0, 28.0]))

shot_logs_2014_15


bucket,attempts,make_rate
enum,u32,f64
"""[0, 4)""",25912,0.6361
"""[4, 10)""",30286,0.4691
"""[10, 16)""",13315,0.4083
"""[16, 22)""",23138,0.4036
"""[22, 28)""",34264,0.3577
"""[28, inf)""",1154,0.1612


hoopr_mirror_2025


bucket,attempts,make_rate
enum,u32,f64
"""[0, 4)""",60263,0.676
"""[4, 10)""",41284,0.4477
"""[10, 16)""",22305,0.4421
"""[16, 22)""",11615,0.4051
"""[22, 28)""",88207,0.3663
"""[28, inf)""",11645,0.3141


hoopr_mirror_2026


bucket,attempts,make_rate
enum,u32,f64
"""[0, 4)""",61524,0.6812
"""[4, 10)""",40127,0.4487
"""[10, 16)""",23344,0.4437
"""[16, 22)""",11958,0.4012
"""[22, 28)""",88166,0.3622
"""[28, inf)""",10995,0.3054


## 6. First look at the defender-distance effect

Unconditioned, closer defenders barely look harmful - because contested shots are disproportionately layups. Conditioning on shot distance is what makes the effect visible, and is how Phase 1 should quantify it.

In [7]:
shot_logs = sources["shot_logs_2014_15"]
display(make_rate_by_bucket(shot_logs, "defender_distance_ft", [0.0, 2.0, 4.0, 6.0]))

with pl.Config(tbl_rows=40):
    display(make_rate_by_defender_and_distance(shot_logs))

bucket,attempts,make_rate
enum,u32,f64
"""[0, 2)""",23468,0.4527
"""[2, 4)""",46524,0.4695
"""[4, 6)""",35178,0.4325
"""[6, inf)""",22899,0.4466


distance_bucket,defender_bucket,attempts,make_rate
enum,enum,u32,f64
"""[0, 4)""","""[0, 2)""",10166,0.5268
"""[0, 4)""","""[2, 4)""",11811,0.6568
"""[0, 4)""","""[4, 6)""",2788,0.8246
"""[0, 4)""","""[6, inf)""",1147,0.9337
"""[4, 10)""","""[0, 2)""",10590,0.4164
"""[4, 10)""","""[2, 4)""",15734,0.4704
"""[4, 10)""","""[4, 6)""",3274,0.577
"""[4, 10)""","""[6, inf)""",688,0.7355
"""[10, 16)""","""[0, 2)""",1395,0.352
